# EcoReCon-AI experiments (ICNGMR 2026 paper)
Recurring-drift module reuse + carbon-aware deferral on a 240-hour simulated edge-camera stream (CIFAR-10, five camera contexts, India grid carbon trace).

**How to run**
1. Runtime → Change runtime type → **T4 GPU** (PyTorch does not use a TPU here; CPU-only would take many hours).
2. Runtime → **Run all**. Allow Google Drive access when asked.
3. Expected time on a T4: about 1–1.5 h (pretraining ~10 min, 50 stream runs, aggregation).
4. **Checkpoints (safe against internet drops / Colab disconnects):** everything is saved on **Google Drive → MyDrive/EcoReCon_results** (results) and **MyDrive/EcoReCon_data** (downloaded data).
   - Pre-training saves a checkpoint after **every epoch** (model, optimiser, learning-rate schedule and random-number states) and resumes from the last finished epoch.
   - Each of the 50 stream runs is saved the moment it finishes; finished runs are skipped.
   - Data downloads are kept on Drive, so they are not repeated.
   - All files are written atomically, so a disconnect cannot leave a broken file.

   **After a disconnect: reconnect, choose T4 GPU again if asked, and press Run all.** It continues where it stopped. (A stream run that was half-way through restarts from its beginning; on a T4 that costs about a minute.)
5. At the end `EcoReCon_results.zip` downloads to your computer automatically.


In [ ]:
# 1. Setup: folders, Google Drive, GPU check
import os, sys, subprocess, hashlib, json, shutil, time
WORK = "/content/ecorecon"; os.makedirs(WORK, exist_ok=True)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/EcoReCon_results"
except Exception as e:
    print("Drive not mounted (", e, ") - results stay in /content only")
    RESULTS = os.path.join(WORK, "results")
DATA = os.path.join(os.path.dirname(RESULTS), "EcoReCon_data") if RESULTS.startswith("/content/drive") else os.path.join(WORK, "data")
for d in (RESULTS, DATA): os.makedirs(d, exist_ok=True)
os.environ.update(RESULTS_DIR=RESULTS, DATA_DIR=DATA, THREADS="2", EPOCHS="15", PYTHONUNBUFFERED="1")
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (CPU only - will be very slow)")
print("results ->", RESULTS, "| data ->", DATA)

def sh(cmd):
    """Run a shell command in the code folder and stream its output live."""
    p = subprocess.Popen(cmd, shell=True, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
    for line in p.stdout: print(line, end="", flush=True)
    if p.wait() != 0: raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


In [ ]:
# 2. Write the experiment source files
SOURCES = {
"common.py": "\"\"\"EcoReCon-AI experiments: shared data, corruption, model and cost utilities (CPU only).\"\"\"\nimport os, pickle, tarfile, json, math\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nDATA = os.environ.get(\"DATA_DIR\", os.path.join(HERE, \"data\"))\nRES = os.environ.get(\"RESULTS_DIR\", os.path.join(HERE, \"results\"))\nDEVICE = torch.device(\"cuda\" if torch.cuda.is_available() and os.environ.get(\"FORCE_CPU\") != \"1\" else \"cpu\")\nos.makedirs(RES, exist_ok=True)\ntorch.set_num_threads(int(os.environ.get(\"THREADS\", 8)))\n\nMEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)\nSTD = np.array([0.2470, 0.2435, 0.2616], dtype=np.float32)\n\n\n# ---------------------------------------------------------------- data\ndef load_cifar():\n    \"\"\"Returns (xtr uint8 N,32,32,3, ytr, xte, yte) from the official python tarball.\"\"\"\n    cache = os.path.join(DATA, \"cifar10.npz\")\n    if os.path.exists(cache):\n        z = np.load(cache)\n        return z[\"xtr\"], z[\"ytr\"], z[\"xte\"], z[\"yte\"]\n    tf = tarfile.open(os.path.join(DATA, \"cifar-10-python.tar.gz\"))\n    def batch(name):\n        d = pickle.load(tf.extractfile(f\"cifar-10-batches-py/{name}\"), encoding=\"bytes\")\n        return d[b\"data\"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1), np.array(d[b\"labels\"])\n    xs, ys = zip(*[batch(f\"data_batch_{i}\") for i in range(1, 6)])\n    xtr, ytr = np.concatenate(xs), np.concatenate(ys)\n    xte, yte = batch(\"test_batch\")\n    np.savez_compressed(cache + \".tmp.npz\", xtr=xtr, ytr=ytr, xte=xte, yte=yte)\n    os.replace(cache + \".tmp.npz\", cache)\n    return xtr, ytr, xte, yte\n\n\n# ---------------------------------------------------------------- recurring contexts (edge camera conditions)\nCONTEXTS = [\"clean\", \"low_light\", \"noise\", \"blur\", \"haze\"]\n\n\ndef _gauss_kernel(sigma, size=7):\n    ax = np.arange(size) - size // 2\n    k = np.exp(-ax ** 2 / (2 * sigma ** 2)); k /= k.sum()\n    return k\n\n\ndef corrupt(x, ctx, rng):\n    \"\"\"x: float32 N,32,32,3 in [0,1]. Fixed mid-level severity per context.\"\"\"\n    if ctx == \"clean\":\n        y = x\n    elif ctx == \"low_light\":          # night-time: dark, low contrast, sensor noise\n        y = 0.30 * x ** 1.2 + rng.normal(0, 0.03, x.shape)\n    elif ctx == \"noise\":              # high-gain / rain-on-lens style sensor noise\n        y = x + rng.normal(0, 0.10, x.shape)\n    elif ctx == \"blur\":               # motion / defocus blur\n        k = _gauss_kernel(1.3)\n        t = torch.from_numpy(x).permute(0, 3, 1, 2)\n        kk = torch.from_numpy(k).float()\n        t = F.conv2d(F.pad(t.reshape(-1, 1, 32, 32), (3, 3, 0, 0), mode=\"reflect\"), kk.view(1, 1, 1, 7))\n        t = F.conv2d(F.pad(t, (0, 0, 3, 3), mode=\"reflect\"), kk.view(1, 1, 7, 1))\n        y = t.reshape(-1, 3, 32, 32).permute(0, 2, 3, 1).numpy()\n    elif ctx == \"haze\":               # fog / dust haze: blend towards grey airlight\n        y = 0.45 * x + 0.55 * 0.75\n    else:\n        raise ValueError(ctx)\n    return np.clip(y, 0, 1).astype(np.float32)\n\n\ndef to_tensor(x01):\n    return torch.from_numpy(((x01 - MEAN) / STD).transpose(0, 3, 1, 2).copy()).to(DEVICE)\n\n\n# ---------------------------------------------------------------- model: ResNet-10-ish, split into shared trunk + reconfigurable module\nclass Block(nn.Module):\n    def __init__(self, cin, cout, stride):\n        super().__init__()\n        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)\n        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)\n        self.sc = None\n        if stride != 1 or cin != cout:\n            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))\n\n    def forward(self, x):\n        o = F.relu(self.b1(self.c1(x)))\n        o = self.b2(self.c2(o))\n        return F.relu(o + (x if self.sc is None else self.sc(x)))\n\n\nclass Net(nn.Module):\n    \"\"\"trunk = stem + stage1 + stage2 (shared, frozen after pre-training);\n    module = stage3 + head (swappable per context). BN layers everywhere.\"\"\"\n    def __init__(self, w=32, ncls=10):\n        super().__init__()\n        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU())\n        self.s1 = nn.Sequential(Block(w, w, 1), Block(w, w, 1))\n        self.s2 = nn.Sequential(Block(w, 2 * w, 2), Block(2 * w, 2 * w, 1))\n        self.s3 = nn.Sequential(Block(2 * w, 4 * w, 2), Block(4 * w, 4 * w, 1))\n        self.head = nn.Linear(4 * w, ncls)\n\n    def trunk(self, x):\n        return self.s2(self.s1(self.stem(x)))\n\n    def module(self, h):\n        h = self.s3(h)\n        return self.head(F.adaptive_avg_pool2d(h, 1).flatten(1))\n\n    def forward(self, x):\n        return self.module(self.trunk(x))\n\n\ndef module_keys(net):\n    \"\"\"state-dict keys that belong to the reconfigurable module (stage3 + head).\"\"\"\n    return [k for k in net.state_dict() if k.startswith((\"s3.\", \"head.\"))]\n\n\ndef bn_keys(net):\n    return [k for k in net.state_dict() if (\"running_\" in k or \"num_batches\" in k)]\n\n\n# ---------------------------------------------------------------- compute cost (MAC counting by forward hooks)\ndef count_macs(net, part=\"all\"):\n    \"\"\"MACs for one 32x32 image: total forward, and the module (stage3+head) part.\"\"\"\n    macs = {\"all\": 0, \"module\": 0}\n    hooks = []\n    def mk(name):\n        def h(m, i, o):\n            if isinstance(m, nn.Conv2d):\n                n = o.shape[1] * o.shape[2] * o.shape[3] * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]\n            elif isinstance(m, nn.Linear):\n                n = m.in_features * m.out_features\n            else:\n                return\n            macs[\"all\"] += n\n            if name.startswith((\"s3\", \"head\")):\n                macs[\"module\"] += n\n        return h\n    for name, m in net.named_modules():\n        if isinstance(m, (nn.Conv2d, nn.Linear)):\n            hooks.append(m.register_forward_hook(mk(name)))\n    net.eval()\n    with torch.no_grad():\n        net(torch.zeros(1, 3, 32, 32, device=next(net.parameters()).device))\n    for h in hooks: h.remove()\n    return macs\n\n\nclass Meter:\n    \"\"\"Counts FLOPs (2 x MACs). Backward ~ 2 x forward for the layers that are trained.\"\"\"\n    def __init__(self, macs):\n        self.f_all = 2 * macs[\"all\"]; self.f_mod = 2 * macs[\"module\"]\n        self.flops = 0.0\n    def infer(self, n):               # forward of whole net\n        self.flops += n * self.f_all\n    def bn_recal(self, n):            # forward only (statistics), whole net\n        self.flops += n * self.f_all\n    def train_module(self, n):        # forward whole + backward through module only\n        self.flops += n * (self.f_all + 2 * self.f_mod)\n    def train_full(self, n):          # forward + backward whole net\n        self.flops += n * 3 * self.f_all\n    def signature(self, n):           # trunk forward is shared with inference; extra cost ~0\n        pass\n\n\ndef save_json(obj, name):\n    path = os.path.join(RES, name)\n    with open(path + \".tmp\", \"w\") as f:\n        json.dump(obj, f, indent=1)\n    os.replace(path + \".tmp\", path)           # atomic: a disconnect never leaves a half-written file\n",
"carbon.py": "\"\"\"Modelled hourly carbon intensity of the Indian grid for 2023 (kg CO2 / kWh).\n\nCI(h) = EF_fossil * (1 - c_other - s(h))\n - s(h): solar share of generation in hour h, proportional to NASA POWER all-sky surface irradiance at\n   Nagpur (21.15 N, 79.09 E, central India, local solar time), scaled so its annual mean equals 7 %\n   (Ember, Global Electricity Review 2025: solar ~7 % of India's 2024 generation).\n - c_other: non-solar clean share (hydro 8 % + wind/nuclear) = 22 % clean - 7 % solar = 15 %, held constant.\n - EF_fossil chosen so the annual mean of CI equals 0.727 kg/kWh (CEA CO2 Baseline Database v20,\n   weighted average for FY 2023-24).\nDemand is assumed flat over the day. This is an estimate of the diurnal shape, not measured data.\n\"\"\"\nimport json, os\nimport numpy as np\nfrom common import DATA, save_json\n\nSOLAR_SHARE, OTHER_CLEAN, MEAN_CI = 0.07, 0.15, 0.727\n\n\ndef build():\n    j = json.load(open(os.path.join(DATA, \"nasa_power_hourly_Nagpur_2023.json\")))\n    v = j[\"properties\"][\"parameter\"][\"ALLSKY_SFC_SW_DWN\"]\n    g = np.array([max(v[k], 0.0) for k in sorted(v)], dtype=float)       # 8760 h, Wh/m2\n    s = SOLAR_SHARE * g / g.mean()\n    fossil = 1 - OTHER_CLEAN - s\n    ef = MEAN_CI / fossil.mean()\n    ci = ef * fossil\n    return ci, ef, s\n\n\nif __name__ == \"__main__\":\n    ci, ef, s = build()\n    np.save(os.path.join(DATA, \"ci_india_2023.npy\"), ci)\n    hod = ci.reshape(365, 24).mean(0)\n    out = dict(ef_fossil=ef, mean=ci.mean(), min=ci.min(), max=ci.max(), p5=np.percentile(ci, 5),\n               p95=np.percentile(ci, 95), peak_solar_share=s.max(),\n               hour_of_day_mean=[round(x, 4) for x in hod])\n    save_json(out, \"carbon_trace_summary.json\")\n    print(json.dumps(out, indent=1))\n",
"pretrain.py": "\"\"\"Pre-train the base network on clean CIFAR-10 (40k images); keep 10k training images as the\n'field pool' that supplies labelled samples during deployment. Saves a checkpoint after every epoch.\"\"\"\nimport sys, time, os\nimport numpy as np, torch, torch.nn as nn, torch.nn.functional as F\nfrom common import load_cifar, to_tensor, Net, RES, save_json, count_macs, DEVICE\n\nseed = int(sys.argv[1]) if len(sys.argv) > 1 else 0\nEPOCHS = int(os.environ.get(\"EPOCHS\", 15))\ntorch.manual_seed(seed); rng = np.random.default_rng(seed)\nxtr, ytr, xte, yte = load_cifar()\nperm = np.random.default_rng(123).permutation(len(xtr))     # split fixed across seeds\npre_idx, field_idx = perm[:40000], perm[40000:]\nnp.save(os.path.join(RES, \"field_idx.npy\"), field_idx)\n_sub = int(os.environ.get(\"PRETRAIN_SUBSET\", 0))             # testing only: train on a subset\nif _sub: pre_idx = pre_idx[:_sub]\nX = xtr[pre_idx].astype(np.float32) / 255; Y = torch.from_numpy(ytr[pre_idx]).long().to(DEVICE)\nXte = to_tensor(xte.astype(np.float32) / 255); Yte = torch.from_numpy(yte).long().to(DEVICE)\n\nnet = Net().to(DEVICE)\nopt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)\nsteps = EPOCHS * (len(X) // 128)\nsched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, total_steps=steps, pct_start=0.25)\nlog = []\n# ---- checkpoint: resume after a disconnect from the last finished epoch (identical result)\nCKPT = os.path.join(RES, f\"ckpt_pretrain_seed{seed}.pt\")\nstart = 0\nif os.path.exists(CKPT):\n    ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)\n    if ck.get(\"epochs\") == EPOCHS and ck.get(\"n_train\") == len(X):\n        net.load_state_dict(ck[\"net\"]); opt.load_state_dict(ck[\"opt\"]); sched.load_state_dict(ck[\"sched\"])\n        rng.bit_generator.state = ck[\"np_rng\"]; torch.set_rng_state(ck[\"torch_rng\"])\n        if torch.cuda.is_available() and ck.get(\"cuda_rng\") is not None: torch.cuda.set_rng_state_all(ck[\"cuda_rng\"])\n        log, start = ck[\"log\"], ck[\"epoch\"]\n        print(f\"resuming pre-training after epoch {start}/{EPOCHS}\", flush=True)\n_stop = int(os.environ.get(\"TEST_INTERRUPT_EPOCH\", 0))      # testing only: simulate a disconnect\nfor ep in range(start, EPOCHS):\n    net.train(); t0 = time.time(); p = rng.permutation(len(X)); tot = 0\n    for b in range(len(X) // 128):\n        idx = p[b * 128:(b + 1) * 128]\n        xb = X[idx]\n        if True:  # augmentation: random crop (pad 4) + horizontal flip\n            xb = np.pad(xb, ((0, 0), (4, 4), (4, 4), (0, 0)), mode=\"reflect\")\n            ox, oy = rng.integers(0, 9, 2)\n            xb = xb[:, oy:oy + 32, ox:ox + 32]\n            flip = rng.random(len(idx)) < 0.5\n            xb[flip] = xb[flip, :, ::-1]\n        out = net(to_tensor(np.ascontiguousarray(xb)))\n        loss = F.cross_entropy(out, Y[idx])\n        opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item()\n    net.eval()\n    with torch.no_grad():\n        acc = sum((net(Xte[i:i + 1000]).argmax(1) == Yte[i:i + 1000]).sum().item() for i in range(0, 10000, 1000)) / 100\n    log.append(dict(epoch=ep + 1, loss=tot / (len(X) // 128), test_acc=acc, sec=time.time() - t0))\n    print(log[-1], flush=True)\n    ck = dict(net=net.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), np_rng=rng.bit_generator.state,\n              torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,\n              log=log, epoch=ep + 1, epochs=EPOCHS, n_train=len(X))\n    torch.save(ck, CKPT + \".tmp\"); os.replace(CKPT + \".tmp\", CKPT)    # atomic\n    if _stop and ep + 1 == _stop:\n        print(\"TEST: simulated disconnect\", flush=True); os._exit(3)\n_bp = os.path.join(RES, f\"base_seed{seed}.pt\")\ntorch.save({k: v.cpu() for k, v in net.state_dict().items()}, _bp + \".tmp\"); os.replace(_bp + \".tmp\", _bp)\nsave_json(dict(seed=seed, epochs=EPOCHS, log=log, params=sum(p.numel() for p in net.parameters()),\n               macs=count_macs(net)), f\"pretrain_seed{seed}.json\")\n",
"stream.py": "\"\"\"Recurring-drift deployment simulator for EcoReCon-AI.\n\nA camera node runs for H hours. Each hour has one operating context (night low-light, haze, blur,\nnoise, clean) that recurs daily. Per hour the node sees an unlabelled batch (used for inference,\nthe context signature and BN recalibration) and receives LAB_PER_H labelled field samples of the\ncurrent context (delayed annotation). Policies decide whether to reuse, adapt now, defer adaptation\nto a low-carbon hour, or expand the module bank. Every policy sees identical data.\n\nUsage: python stream.py <stream_seed> [policy ...]     (env D_DEFER, CAP, PRETRAIN_SEED)\n\"\"\"\nimport sys, os, copy, time, json\nimport numpy as np, torch, torch.nn.functional as F\nfrom common import (load_cifar, corrupt, to_tensor, Net, CONTEXTS, module_keys, count_macs, Meter, RES, DATA,\n                    save_json, DEVICE)\n\nH = int(os.environ.get(\"HOURS\", 240))\nEVAL_N, LAB_PER_H, N_MIN, BUF = 400, 64, 128, 1024\nEPOCHS_MOD, EPOCHS_FULL = 4, 2\nD_DEFER = int(os.environ.get(\"D_DEFER\", 6))\nCAP = int(os.environ.get(\"CAP\", 8))\nEPS_J_PER_FLOP = 1e-9          # assumed edge-CPU efficiency: 1 GFLOP per joule (only scales absolute numbers)\nPRE_SEED = int(os.environ.get(\"PRETRAIN_SEED\", 0))\n\nALL_POLICIES = [\"static\", \"bn_adapt\", \"periodic_full\", \"drift_ft\", \"drift_ft_defer\", \"bank\", \"ecorecon\"]\n\n\n# ------------------------------------------------------------------ stream construction\ndef make_stream(seed):\n    rng = np.random.default_rng(1000 + seed)\n    start_day = int(rng.integers(0, 365 - H // 24 - 1))\n    ci_year = np.load(os.path.join(DATA, \"ci_india_2023.npy\"))\n    ci = ci_year[start_day * 24: start_day * 24 + H]\n    ci_prev = ci_year[start_day * 24 - 24: start_day * 24 + H - 24] if start_day > 0 else np.roll(ci, 24)\n    ctx = []\n    day_ctx = \"clean\"\n    for h in range(H):\n        hod = h % 24\n        if hod >= 19 or hod < 6:                       # night\n            c = \"low_light\" if rng.random() < 0.9 else \"noise\"\n        else:\n            if hod == 6 or rng.random() < 0.25:        # day episodes: change ~every 4 h\n                day_ctx = rng.choice([\"clean\", \"haze\", \"blur\", \"noise\"], p=[0.45, 0.25, 0.15, 0.15])\n                if hod in (6, 7) and rng.random() < 0.5:\n                    day_ctx = \"haze\"                   # morning haze/fog\n            c = day_ctx\n        ctx.append(str(c))\n    return dict(start_day=start_day, ci=ci, ci_forecast=ci_prev, ctx=ctx, rng_seed=seed)\n\n\nclass Data:\n    def __init__(self, seed):\n        xtr, ytr, xte, yte = load_cifar()\n        field = np.load(os.path.join(RES, \"field_idx.npy\"))\n        self.xf, self.yf = xtr[field].astype(np.float32) / 255, ytr[field]\n        self.xt, self.yt = xte.astype(np.float32) / 255, yte\n        self.rng = np.random.default_rng(2000 + seed)\n\n    def hour(self, ctx):\n        i = self.rng.choice(len(self.xt), EVAL_N, replace=False)\n        j = self.rng.choice(len(self.xf), LAB_PER_H, replace=False)\n        xe = to_tensor(corrupt(self.xt[i], ctx, self.rng)); ye = torch.from_numpy(self.yt[i]).long().to(DEVICE)\n        xl = to_tensor(corrupt(self.xf[j], ctx, self.rng)); yl = torch.from_numpy(self.yf[j]).long().to(DEVICE)\n        return xe, ye, xl, yl\n\n\n# ------------------------------------------------------------------ model helpers\ndef set_trunk_bn_train(net, on):\n    for m in [net.stem, net.s1, net.s2, net.s3]:\n        for mod in m.modules():\n            if isinstance(mod, torch.nn.BatchNorm2d):\n                mod.train(on)\n                if on:\n                    mod.momentum = None            # cumulative average over the given data\n                    mod.reset_running_stats()\n\n\ndef bn_recal(net, x, meter):\n    set_trunk_bn_train(net, True)\n    with torch.no_grad():\n        for i in range(0, len(x), 200):\n            net(x[i:i + 200])\n    net.eval(); meter.bn_recal(len(x))\n\n\ndef accuracy(net, x, y, meter):\n    net.eval()\n    with torch.no_grad():\n        p = net(x).argmax(1)\n    meter.infer(len(x))\n    return (p == y).float().mean().item()\n\n\ndef train_module(net, x, y, meter, epochs=EPOCHS_MOD, seed=0):\n    \"\"\"Fine-tune stage3 + head on labelled context data; trunk BN statistics recalibrated first.\"\"\"\n    g = torch.Generator().manual_seed(seed)\n    bn_recal(net, x, meter)\n    for p in net.parameters(): p.requires_grad_(False)\n    params = list(net.s3.parameters()) + list(net.head.parameters())\n    for p in params: p.requires_grad_(True)\n    opt = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=5e-4)\n    net.eval(); net.s3.train()                     # trunk frozen incl. its BN stats; module BN adapts\n    for _ in range(epochs):\n        perm = torch.randperm(len(x), generator=g)\n        for b in range(0, len(x), 64):\n            idx = perm[b:b + 64]\n            loss = F.cross_entropy(net(x[idx]), y[idx])\n            opt.zero_grad(); loss.backward(); opt.step()\n    meter.train_module(len(x) * epochs)\n    for p in net.parameters(): p.requires_grad_(True)\n    net.eval()\n\n\ndef train_full(net, x, y, meter, epochs=EPOCHS_FULL, seed=0):\n    g = torch.Generator().manual_seed(seed)\n    opt = torch.optim.SGD(net.parameters(), lr=0.005, momentum=0.9, weight_decay=5e-4)\n    net.train()\n    for _ in range(epochs):\n        perm = torch.randperm(len(x), generator=g)\n        for b in range(0, len(x), 64):\n            idx = perm[b:b + 64]\n            loss = F.cross_entropy(net(x[idx]), y[idx])\n            opt.zero_grad(); loss.backward(); opt.step()\n    meter.train_full(len(x) * epochs)\n    net.eval()\n\n\n# ------------------------------------------------------------------ label-free context signature\nclass Probe:\n    \"\"\"Per-channel mean/std of the frozen base trunk output (clean BN statistics).\"\"\"\n    def __init__(self, base, meter_trunk_flops):\n        self.net = copy.deepcopy(base).eval(); self.f = meter_trunk_flops\n\n    def sig(self, x, meter):\n        with torch.no_grad():\n            h = self.net.trunk(x)                     # N,64,8,8\n        meter.flops += len(x) * self.f\n        m = h.mean((0, 2, 3)); s = h.std((0, 2, 3)) + 1e-4\n        return torch.cat([m, s.log()])\n\n    @staticmethod\n    def dist(a, b):\n        return (a - b).abs().mean().item()\n\n\n# ------------------------------------------------------------------ policies\ndef best_slot(stream, t, deadline):\n    \"\"\"Hour in [t, deadline] with the lowest forecast carbon intensity (forecast = same hour yesterday).\"\"\"\n    hi = min(deadline, H - 1)\n    cand = range(t, hi + 1)\n    return min(cand, key=lambda k: stream[\"ci_forecast\"][k])\n\n\ndef run(policy, stream, data_seed, base_state, tau, macs, trunk_flops, d_defer=D_DEFER, cap=CAP):\n    torch.manual_seed(0)\n    data = Data(data_seed)\n    meter = Meter(macs)\n    adapt_flops_t = np.zeros(H)                      # adaptation compute (excl. plain inference) per hour\n    infer_flops_t = np.zeros(H)\n    base = Net().to(DEVICE); base.load_state_dict(base_state); base.eval()\n    probe = Probe(base, trunk_flops)\n    net = copy.deepcopy(base)\n    bank = []                                        # list of dict(sig, state, ctx_seen, last_used)\n    buf = {}                                         # labelled buffer per detected cluster id\n    pending = None                                   # dict(cluster, due, deadline)\n    cur_cluster = None\n    rec = []\n    trained_events = 0\n    sig_last = None\n    hist_x, hist_y = [], []\n\n    for t in range(H):\n        ctx = stream[\"ctx\"][t]\n        xe, ye, xl, yl = data.hour(ctx)\n        f0 = meter.flops\n        action = \"serve\"\n\n        if policy == \"static\":\n            pass\n\n        elif policy == \"bn_adapt\":                      # test-time BN recalibration every hour, label-free\n            bn_recal(net, xe, meter); action = \"bn\"\n\n        elif policy == \"periodic_full\":                  # full fine-tune at midnight on the last 24 h of labels\n            hist_x.append(xl); hist_y.append(yl)\n            hist_x, hist_y = hist_x[-24:], hist_y[-24:]\n            if t % 24 == 0 and t > 0:\n                train_full(net, torch.cat(hist_x), torch.cat(hist_y), meter, seed=t); trained_events += 1\n                action = \"train_full\"\n\n        elif policy in (\"drift_ft\", \"drift_ft_defer\"):   # single model, threshold-triggered in-place retrain\n            s = probe.sig(xe, meter)\n            if sig_last is None:\n                sig_last = probe.sig(to_tensor(corrupt(data.xf[:400], \"clean\", np.random.default_rng(0))), meter)\n            drift = Probe.dist(s, sig_last) > tau\n            if drift and pending is None:\n                pending = dict(sig=s, start=t, xs=[], ys=[], due=None)\n                bn_recal(net, xe, meter); action = \"bn\"\n            if pending is not None:\n                if Probe.dist(s, pending[\"sig\"]) <= tau:\n                    pending[\"xs\"].append(xl); pending[\"ys\"].append(yl)\n                elif pending[\"due\"] is None:              # context moved on before a job was booked: restart\n                    pending = dict(sig=s, start=t, xs=[xl], ys=[yl], due=None)\n                # a booked (deferred) job is kept and runs at its due hour on the labels it already holds\n                n = sum(len(v) for v in pending[\"xs\"])\n                if n >= N_MIN and pending[\"due\"] is None:\n                    pending[\"due\"] = t if policy == \"drift_ft\" else best_slot(stream, t, t + d_defer)\n                if pending[\"due\"] is not None and t >= pending[\"due\"]:\n                    x = torch.cat(pending[\"xs\"])[-BUF:]; y = torch.cat(pending[\"ys\"])[-BUF:]\n                    train_module(net, x, y, meter, seed=t); trained_events += 1\n                    sig_last = pending[\"sig\"]; pending = None; action = \"train\"\n                elif action == \"serve\":\n                    bn_recal(net, xe, meter); action = \"bn\"\n\n        elif policy in (\"bank\", \"ecorecon\"):\n            s = probe.sig(xe, meter)\n            if not bank:                                  # base model = first module (clean context)\n                s0 = probe.sig(to_tensor(corrupt(data.xf[:400], \"clean\", np.random.default_rng(0))), meter)\n                bank.append(dict(sig=s0, state=copy.deepcopy(base.state_dict()), last=0, n_train=0))\n            if pending is None:\n                pending = []                              # list of expansion jobs (one per unknown context)\n            d = [Probe.dist(s, m[\"sig\"]) for m in bank]\n            k = int(np.argmin(d))\n            if d[k] <= tau:                               # REUSE a stored module (no training)\n                net.load_state_dict(bank[k][\"state\"]); bank[k][\"last\"] = t; action = \"reuse\"\n            else:                                         # unknown context -> serve nearest + BN recal, feed EXPAND job\n                net.load_state_dict(bank[k][\"state\"]); bn_recal(net, xe, meter); action = \"bn\"\n                dj = [Probe.dist(s, j[\"sig\"]) for j in pending]\n                if dj and min(dj) <= tau:\n                    job = pending[int(np.argmin(dj))]\n                else:\n                    job = dict(sig=s, start=t, xs=[], ys=[], due=None, warm=k); pending.append(job)\n                job[\"xs\"].append(xl); job[\"ys\"].append(yl)\n            for job in list(pending):\n                n = sum(len(v) for v in job[\"xs\"])\n                if n >= N_MIN and job[\"due\"] is None:\n                    job[\"due\"] = t if policy == \"bank\" else best_slot(stream, t, t + d_defer)\n                if job[\"due\"] is not None and t >= job[\"due\"]:\n                    x = torch.cat(job[\"xs\"])[-BUF:]; y = torch.cat(job[\"ys\"])[-BUF:]\n                    new = copy.deepcopy(base); new.load_state_dict(bank[job[\"warm\"]][\"state\"])\n                    train_module(new, x, y, meter, seed=t); trained_events += 1\n                    entry = dict(sig=job[\"sig\"], state=copy.deepcopy(new.state_dict()), last=t, n_train=1)\n                    if len(bank) >= cap:                  # evict least recently used (never the base module 0)\n                        j = 1 + int(np.argmin([m[\"last\"] for m in bank[1:]]))\n                        bank.pop(j)\n                    bank.append(entry); pending[:] = [j_ for j_ in pending if j_ is not job]; action = \"train\"   # remove by identity (dicts hold tensors)\n                    if Probe.dist(s, entry[\"sig\"]) <= tau:   # current hour belongs to the new module: serve it\n                        net.load_state_dict(entry[\"state\"])\n        else:\n            raise ValueError(policy)\n\n        adapt_flops_t[t] = meter.flops - f0\n        acc = accuracy(net, xe, ye, meter)\n        infer_flops_t[t] = 2 * macs[\"all\"] * len(xe)\n        rec.append(dict(t=t, ctx=ctx, acc=acc, action=action, ci=float(stream[\"ci\"][t])))\n\n    accs = np.array([r[\"acc\"] for r in rec])\n    co2_adapt = float((adapt_flops_t * EPS_J_PER_FLOP / 3.6e6 * stream[\"ci\"]).sum() * 1000)   # grams\n    co2_total = float(((adapt_flops_t + infer_flops_t) * EPS_J_PER_FLOP / 3.6e6 * stream[\"ci\"]).sum() * 1000)\n    out = dict(policy=policy, mean_acc=float(accs.mean()),\n               acc_by_ctx={c: float(np.mean([r[\"acc\"] for r in rec if r[\"ctx\"] == c])) for c in CONTEXTS\n                           if any(r[\"ctx\"] == c for r in rec)},\n               adapt_gflop=float(adapt_flops_t.sum() / 1e9), co2_adapt_g=co2_adapt, co2_total_g=co2_total,\n               mean_ci_of_adapt=float((adapt_flops_t * stream[\"ci\"]).sum() / max(adapt_flops_t.sum(), 1)),\n               n_train=trained_events, bank_size=len(bank) if bank else None, records=rec)\n    return out\n\n\ndef calibrate_tau(base_state, trunk_flops, macs):\n    \"\"\"tau from clean data only: 3x the 99th percentile of clean batch-to-batch signature distances.\"\"\"\n    base = Net().to(DEVICE); base.load_state_dict(base_state)\n    probe = Probe(base, trunk_flops); meter = Meter(macs)\n    xtr, ytr, xte, yte = load_cifar()\n    field = np.load(os.path.join(RES, \"field_idx.npy\"))\n    xf = xtr[field].astype(np.float32) / 255\n    rng = np.random.default_rng(7)\n    ref = probe.sig(to_tensor(xf[:400]), meter)\n    ds = [Probe.dist(probe.sig(to_tensor(xf[rng.choice(len(xf), EVAL_N, replace=False)]), meter), ref) for _ in range(200)]\n    return 3 * float(np.percentile(ds, 99)), ds\n\n\nif __name__ == \"__main__\":\n    seed = int(sys.argv[1])\n    pols = sys.argv[2:] or ALL_POLICIES\n    base_state = torch.load(os.path.join(RES, f\"base_seed{PRE_SEED}.pt\"), map_location=DEVICE)\n    tmp = Net(); tmp.load_state_dict(base_state)\n    macs = count_macs(tmp)\n    trunk_flops = 2 * (macs[\"all\"] - macs[\"module\"])\n    tau_file = os.path.join(RES, f\"tau_pre{PRE_SEED}.json\")\n    if os.path.exists(tau_file):\n        tau = json.load(open(tau_file))[\"tau\"]\n    else:\n        tau, ds = calibrate_tau(base_state, trunk_flops, macs)\n        save_json(dict(tau=tau, clean_p99=float(np.percentile(ds, 99)), clean_max=float(max(ds))), os.path.basename(tau_file))\n    stream = make_stream(seed)\n    tag = f\"s{seed}_pre{PRE_SEED}_D{D_DEFER}_C{CAP}\"\n    for p in pols:\n        t0 = time.time()\n        r = run(p, stream, seed, base_state, tau, macs, trunk_flops)\n        r.update(seed=seed, pre_seed=PRE_SEED, d_defer=D_DEFER, cap=CAP, tau=tau, start_day=stream[\"start_day\"],\n                 sec=time.time() - t0)\n        save_json(r, f\"run_{tag}_{p}.json\")\n        print(f\"{tag} {p:15s} acc={r['mean_acc']:.4f} adaptGF={r['adapt_gflop']:.1f} co2a={r['co2_adapt_g']:.4f}g \"\n              f\"ciAdapt={r['mean_ci_of_adapt']:.3f} ntrain={r['n_train']} bank={r['bank_size']} {r['sec']:.0f}s\", flush=True)\n",
"diag_tau.py": "\"\"\"Diagnostic: are contexts separable by the label-free signature at the clean-calibrated tau?\nReports within-context and between-context signature distances, and base accuracy per context.\"\"\"\nimport os, json, numpy as np, torch\nfrom common import load_cifar, corrupt, to_tensor, Net, CONTEXTS, count_macs, Meter, RES, save_json, DEVICE\nfrom stream import Probe, calibrate_tau, EVAL_N\n\nbase_state = torch.load(os.path.join(RES, \"base_seed0.pt\"), map_location=DEVICE)\nnet = Net().to(DEVICE); net.load_state_dict(base_state); net.eval()\nmacs = count_macs(net); tf = 2 * (macs[\"all\"] - macs[\"module\"])\ntau, ds = calibrate_tau(base_state, tf, macs)\nsave_json(dict(tau=tau, clean_p99=float(np.percentile(ds, 99)), clean_max=float(max(ds))), \"tau_pre0.json\")\nprobe = Probe(net, tf); meter = Meter(macs)\nxtr, ytr, xte, yte = load_cifar()\nxt = xte.astype(np.float32) / 255\nrng = np.random.default_rng(3)\nsigs = {c: [] for c in CONTEXTS}; acc = {}\nfor c in CONTEXTS:\n    for r in range(8):\n        i = rng.choice(len(xt), EVAL_N, replace=False)\n        x = to_tensor(corrupt(xt[i], c, rng))\n        sigs[c].append(probe.sig(x, meter))\n        if r == 0:\n            with torch.no_grad():\n                acc[c] = (net(x).argmax(1).cpu().numpy() == yte[i]).mean()\nwithin = {c: float(np.max([Probe.dist(a, b) for a in sigs[c] for b in sigs[c] if a is not b])) for c in CONTEXTS}\nbetween = {f\"{a}-{b}\": float(np.min([Probe.dist(x, y) for x in sigs[a] for y in sigs[b]]))\n           for i, a in enumerate(CONTEXTS) for b in CONTEXTS[i + 1:]}\nout = dict(tau=tau, within_max=within, between_min=between, base_acc={k: float(v) for k, v in acc.items()})\nsave_json(out, \"diag_tau.json\"); print(json.dumps(out, indent=1))\n",
"make_jobs.py": "\"\"\"Write jobs.txt: one line per (env, seed, policy) stream run. Main grid first, then sensitivity.\"\"\"\nimport os\nfrom stream import ALL_POLICIES\nlines = []\nfor s in range(5):\n    for p in ALL_POLICIES:\n        lines.append(f\"D_DEFER=6 CAP=8 {s} {p}\")\nfor d in (3, 12, 24):                 # D=0 is identical to 'bank' by construction (best slot of [t,t] is t)\n    for s in range(3):                # shorter plan (user, 2026-11-11): sensitivity on seeds 0-2\n        lines.append(f\"D_DEFER={d} CAP=8 {s} ecorecon\")\nfor c in (2,):                        # shorter plan: CAP=2 only\n    for s in range(3):\n        for p in (\"bank\", \"ecorecon\"):\n            lines.append(f\"D_DEFER=6 CAP={c} {s} {p}\")\nopen(os.path.join(os.path.dirname(os.path.abspath(__file__)), \"jobs.txt\"), \"w\").write(\"\\n\".join(lines) + \"\\n\")\nprint(len(lines), \"jobs\")\n",
"run_all.py": "\"\"\"Run every job in jobs.txt inside ONE process (Colab-friendly). Each finished run is saved at once as\nresults/run_s{seed}_pre0_D{D}_C{C}_{policy}.json; finished runs are skipped when restarted.\"\"\"\nimport os, json, time, sys\nimport numpy as np, torch\nimport stream\nfrom stream import make_stream, run, calibrate_tau\nfrom common import Net, count_macs, RES, save_json, DEVICE\n\njobs = [l.split() for l in open(os.environ.get(\"JOBS_FILE\", os.path.join(os.path.dirname(os.path.abspath(__file__)), \"jobs.txt\"))) if l.strip()]\nbase_state = torch.load(os.path.join(RES, \"base_seed0.pt\"), map_location=DEVICE)\ntmp = Net().to(DEVICE); tmp.load_state_dict(base_state); macs = count_macs(tmp)\ntrunk_flops = 2 * (macs[\"all\"] - macs[\"module\"])\ntau_file = os.path.join(RES, \"tau_pre0.json\")\nif os.path.exists(tau_file):\n    tau = json.load(open(tau_file))[\"tau\"]\nelse:\n    tau, ds = calibrate_tau(base_state, trunk_flops, macs)\n    save_json(dict(tau=tau, clean_p99=float(np.percentile(ds, 99)), clean_max=float(max(ds))), \"tau_pre0.json\")\nprint(\"device\", DEVICE, \"tau\", round(tau, 4), \"jobs\", len(jobs), flush=True)\nt_all = time.time(); done = 0\nfor k, (dd, cc, s, p) in enumerate(jobs):\n    d = int(dd.split(\"=\")[1]); c = int(cc.split(\"=\")[1]); s = int(s)\n    tag = f\"s{s}_pre0_D{d}_C{c}\"\n    out = os.path.join(RES, f\"run_{tag}_{p}.json\")\n    if os.path.exists(out):\n        continue\n    t0 = time.time()\n    r = run(p, make_stream(s), s, base_state, tau, macs, trunk_flops, d_defer=d, cap=c)\n    r.update(seed=s, pre_seed=0, d_defer=d, cap=c, tau=tau, start_day=make_stream(s)[\"start_day\"], sec=time.time() - t0,\n             device=str(DEVICE))\n    save_json(r, os.path.basename(out)); done += 1\n    el = time.time() - t_all\n    print(f\"[{k+1}/{len(jobs)}] {tag} {p:15s} acc={r['mean_acc']:.4f} adaptGF={r['adapt_gflop']:.1f} \"\n          f\"co2a={r['co2_adapt_g']:.4f}g ntrain={r['n_train']} bank={r['bank_size']} {r['sec']:.0f}s | elapsed {el/60:.1f} min\",\n          flush=True)\nprint(\"ALL_JOBS_FINISHED\", flush=True)\n",
"aggregate.py": "\"\"\"Aggregate stream results -> tables (json/csv/md) and figures for the EcoReCon paper.\"\"\"\nimport os, glob, json, collections\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch\n\nHERE = os.path.dirname(os.path.abspath(__file__)); RES = os.environ.get(\"RESULTS_DIR\", os.path.join(HERE, \"results\"))\nFIG = os.path.join(RES, \"figs\"); os.makedirs(FIG, exist_ok=True)\nplt.rcParams.update({\"font.family\": \"serif\", \"font.serif\": [\"Times New Roman\", \"DejaVu Serif\"], \"font.size\": 8,\n                     \"axes.titlesize\": 8, \"axes.labelsize\": 8, \"legend.fontsize\": 7, \"savefig.dpi\": 300,\n                     \"savefig.bbox\": \"tight\"})\nOI = [\"#000000\", \"#E69F00\", \"#56B4E9\", \"#009E73\", \"#F0E442\", \"#0072B2\", \"#D55E00\", \"#CC79A7\"]\nPOL = [\"static\", \"bn_adapt\", \"periodic_full\", \"drift_ft\", \"drift_ft_defer\", \"bank\", \"ecorecon\"]\nNAME = {\"static\": \"Static\", \"bn_adapt\": \"BN-adapt\", \"periodic_full\": \"Periodic-full\", \"drift_ft\": \"Drift-FT\",\n        \"drift_ft_defer\": \"Drift-FT-defer\", \"bank\": \"Bank\", \"ecorecon\": \"EcoReCon\"}\nCTX = [\"clean\", \"low_light\", \"noise\", \"blur\", \"haze\"]\n\n\ndef load(d=6, c=8):\n    out = collections.defaultdict(dict)\n    for f in glob.glob(os.path.join(RES, f\"run_s*_pre0_D{d}_C{c}_*.json\")):\n        r = json.load(open(f)); out[r[\"policy\"]][r[\"seed\"]] = r\n    return out\n\n\ndef retrain_on_recurrence(r):\n    \"\"\"Training events whose context had already been trained on earlier in the stream.\"\"\"\n    seen, rep, tot = set(), 0, 0\n    for x in r[\"records\"]:\n        if x[\"action\"] in (\"train\", \"train_full\"):\n            tot += 1\n            if x[\"ctx\"] in seen: rep += 1\n            seen.add(x[\"ctx\"])\n    return rep, tot\n\n\ndef ms(v):\n    v = np.asarray(v, float); return float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0\n\n\nmain = load()\nseeds = sorted(set.intersection(*[set(main[p]) for p in POL if p in main])) if all(p in main for p in POL) else []\nprint(\"policies:\", {p: sorted(main[p]) for p in main}, \"common seeds:\", seeds)\nsumm = {}\nfor p in POL:\n    if p not in main: continue\n    S = [main[p][s] for s in seeds] if seeds else list(main[p].values())\n    reps = [retrain_on_recurrence(r) for r in S]\n    acts = collections.Counter(x[\"action\"] for r in S for x in r[\"records\"])\n    summ[p] = dict(n=len(S), acc=ms([r[\"mean_acc\"] for r in S]), gflop=ms([r[\"adapt_gflop\"] for r in S]),\n                   co2=ms([r[\"co2_adapt_g\"] for r in S]), co2_total=ms([r[\"co2_total_g\"] for r in S]),\n                   ci=ms([r[\"mean_ci_of_adapt\"] for r in S if r[\"adapt_gflop\"] > 0] or [0]),\n                   ntrain=ms([r[\"n_train\"] for r in S]), retrain_rep=ms([a for a, b in reps]),\n                   bank=ms([r[\"bank_size\"] or 0 for r in S]),\n                   acc_ctx={c: ms([r[\"acc_by_ctx\"].get(c, np.nan) for r in S]) for c in CTX},\n                   actions={k: v / len(S) for k, v in acts.items()},\n                   per_seed=dict(acc=[r[\"mean_acc\"] for r in S], co2=[r[\"co2_adapt_g\"] for r in S],\n                                 gflop=[r[\"adapt_gflop\"] for r in S], ntrain=[r[\"n_train\"] for r in S]))\n# paired differences vs EcoReCon\npairs = {}\nif \"ecorecon\" in summ:\n    E = summ[\"ecorecon\"][\"per_seed\"]\n    for p in (\"drift_ft\", \"drift_ft_defer\", \"bank\", \"periodic_full\", \"bn_adapt\"):\n        if p not in summ: continue\n        O = summ[p][\"per_seed\"]\n        pairs[p] = dict(d_acc_pp=[100 * (a - b) for a, b in zip(E[\"acc\"], O[\"acc\"])],\n                        co2_ratio=[a / b if b else None for a, b in zip(E[\"co2\"], O[\"co2\"])],\n                        gflop_ratio=[a / b if b else None for a, b in zip(E[\"gflop\"], O[\"gflop\"])])\n# context occupancy\nctx_share = collections.Counter(x[\"ctx\"] for s in seeds for x in main[\"static\"][s][\"records\"]) if \"static\" in main else {}\ntot = sum(ctx_share.values()) or 1\n# sensitivity\nsens = {}\nfor d in (3, 6, 12, 24):\n    R = load(d, 8).get(\"ecorecon\", {})\n    if R: sens[f\"D{d}\"] = dict(n=len(R), acc=ms([r[\"mean_acc\"] for r in R.values()]), co2=ms([r[\"co2_adapt_g\"] for r in R.values()]),\n                              ci=ms([r[\"mean_ci_of_adapt\"] for r in R.values()]))\nfor c in (2, 3, 8):\n    for p in (\"bank\", \"ecorecon\"):\n        R = load(6, c).get(p, {})\n        if R: sens[f\"C{c}_{p}\"] = dict(n=len(R), acc=ms([r[\"mean_acc\"] for r in R.values()]), co2=ms([r[\"co2_adapt_g\"] for r in R.values()]),\n                                      ntrain=ms([r[\"n_train\"] for r in R.values()]))\ntau = json.load(open(os.path.join(RES, \"tau_pre0.json\")))\npre = json.load(open(os.path.join(RES, \"pretrain_seed0.json\")))\ndiag = json.load(open(os.path.join(RES, \"diag_tau.json\")))\nout = dict(seeds=seeds, summary=summ, pairs=pairs, ctx_share={k: v / tot for k, v in ctx_share.items()}, sensitivity=sens,\n           tau=tau, pretrain=dict(final_acc=pre[\"log\"][-1][\"test_acc\"], params=pre[\"params\"], macs=pre[\"macs\"]), diag=diag)\njson.dump(out, open(os.path.join(RES, \"aggregate.json\"), \"w\"), indent=1)\n\n# ---- markdown table preview\nlines = [\"| Policy | Acc. (%) | Adapt. GFLOP | Adapt. CO2 (g) | Mean CI of adapt. | Train events | Re-trains on recurrence |\"]\nfor p in POL:\n    if p not in summ: continue\n    s = summ[p]\n    lines.append(f\"| {NAME[p]} | {100*s['acc'][0]:.1f} \u00b1 {100*s['acc'][1]:.1f} | {s['gflop'][0]:.0f} \u00b1 {s['gflop'][1]:.0f} | \"\n                 f\"{s['co2'][0]:.3f} \u00b1 {s['co2'][1]:.3f} | {s['ci'][0]:.3f} | {s['ntrain'][0]:.1f} | {s['retrain_rep'][0]:.1f} |\")\nopen(os.path.join(RES, \"table_main.md\"), \"w\").write(\"\\n\".join(lines) + \"\\n\")\nprint(\"\\n\".join(lines))\nfor p, v in pairs.items():\n    print(\"EcoReCon vs\", p, \"dAcc pp\", np.round(v[\"d_acc_pp\"], 2), \"CO2 ratio\", np.round([x for x in v[\"co2_ratio\"] if x], 3))\nprint(\"sens\", json.dumps(sens, indent=0)[:1500])\n\n# ---- Fig 2: accuracy vs adaptation CO2\nif summ:\n    fig, ax = plt.subplots(figsize=(3.4, 2.5))\n    for i, p in enumerate(POL):\n        if p not in summ: continue\n        s = summ[p]\n        ax.errorbar(s[\"co2\"][0] * 1000, 100 * s[\"acc\"][0], xerr=s[\"co2\"][1] * 1000, yerr=100 * s[\"acc\"][1],\n                    fmt=\"o\", ms=4, color=OI[i], capsize=2, label=NAME[p])\n    ax.set_xlabel(\"Adaptation emissions (mg CO$_2$, 240 h)\"); ax.set_ylabel(\"Mean accuracy (%)\")\n    ax.legend(frameon=False, fontsize=6.5, loc=\"lower right\"); ax.grid(alpha=0.25, lw=0.5)\n    fig.savefig(os.path.join(FIG, \"fig2_acc_vs_co2.png\")); plt.close(fig)\n\n# ---- Fig 3: timeline, seed 0, first 72 h\nif seeds and \"ecorecon\" in main and \"drift_ft\" in main:\n    s0 = seeds[0]; T = 72\n    fig, ax = plt.subplots(4, 1, figsize=(7.0, 4.2), sharex=True, gridspec_kw=dict(height_ratios=[0.5, 1, 0.8, 1.4]))\n    rec = main[\"ecorecon\"][s0][\"records\"][:T]\n    cmap = {c: col for c, col in zip(CTX, [\"#FFFFFF\", \"#333333\", \"#999999\", \"#56B4E9\", \"#E69F00\"])}\n    for x in rec:\n        ax[0].axvspan(x[\"t\"], x[\"t\"] + 1, color=cmap[x[\"ctx\"]], lw=0)\n    ax[0].set_yticks([]); ax[0].set_ylabel(\"Context\", rotation=0, ha=\"right\", va=\"center\")\n    from matplotlib.patches import Patch\n    ax[0].legend(handles=[Patch(facecolor=cmap[c], edgecolor=\"0.5\", label=c.replace(\"_\", \" \")) for c in CTX], ncol=5,\n                 frameon=False, fontsize=6.5, loc=\"lower center\", bbox_to_anchor=(0.5, 1.0))\n    ax[1].plot([x[\"t\"] + 0.5 for x in rec], [x[\"ci\"] for x in rec], color=\"k\", lw=1)\n    ax[1].set_ylabel(\"CI (kg/kWh)\")\n    for j, p in enumerate([\"drift_ft\", \"bank\", \"ecorecon\"]):\n        rr = main[p][s0][\"records\"][:T]\n        for x in rr:\n            if x[\"action\"] in (\"train\",):\n                ax[2].plot(x[\"t\"] + 0.5, j, \"v\", color=OI[[1, 5, 6][j]], ms=5)\n            elif x[\"action\"] == \"reuse\":\n                ax[2].plot(x[\"t\"] + 0.5, j, \"|\", color=\"0.6\", ms=5)\n    ax[2].set_yticks([0, 1, 2]); ax[2].set_yticklabels([\"Drift-FT\", \"Bank\", \"EcoReCon\"]); ax[2].set_ylim(-0.6, 2.6)\n    for j, p in enumerate([\"static\", \"drift_ft\", \"ecorecon\"]):\n        rr = main[p][s0][\"records\"][:T]\n        ax[3].step([x[\"t\"] for x in rr], [100 * x[\"acc\"] for x in rr], where=\"post\", color=OI[[0, 1, 6][j]], lw=0.9, label=NAME[p])\n    ax[3].set_ylabel(\"Accuracy (%)\"); ax[3].set_xlabel(\"Hour of stream\"); ax[3].legend(frameon=False, ncol=3, fontsize=6.5)\n    for a in ax: a.set_xlim(0, T)\n    fig.savefig(os.path.join(FIG, \"fig3_timeline.png\")); plt.close(fig)\n\n# ---- Fig 4: sensitivity\nif sens:\n    fig, ax = plt.subplots(1, 2, figsize=(7.0, 2.3))\n    ds = [d for d in (3, 6, 12, 24) if f\"D{d}\" in sens]\n    if ds:\n        a2 = ax[0].twinx()\n        ax[0].errorbar(ds, [100 * sens[f\"D{d}\"][\"acc\"][0] for d in ds], yerr=[100 * sens[f\"D{d}\"][\"acc\"][1] for d in ds], fmt=\"-o\", color=OI[5], ms=3, capsize=2)\n        a2.errorbar(ds, [1000 * sens[f\"D{d}\"][\"co2\"][0] for d in ds], yerr=[1000 * sens[f\"D{d}\"][\"co2\"][1] for d in ds], fmt=\"--s\", color=OI[6], ms=3, capsize=2)\n        if \"bank\" in summ:\n            ax[0].axhline(100 * summ[\"bank\"][\"acc\"][0], color=OI[5], lw=0.6, ls=\":\")\n            a2.axhline(1000 * summ[\"bank\"][\"co2\"][0], color=OI[6], lw=0.6, ls=\":\")\n        ax[0].set_xlabel(\"Deferral window D (h)\"); ax[0].set_ylabel(\"Accuracy (%)\", color=OI[5]); a2.set_ylabel(\"Adapt. CO$_2$ (mg)\", color=OI[6])\n        ax[0].set_xticks(ds); ax[0].set_title(\"(a) Deferral window (dotted: Bank, D = 0)\")\n    cs = [c for c in (2, 3, 8) if f\"C{c}_ecorecon\" in sens]\n    if cs:\n        w = 0.35; x = np.arange(len(cs))\n        for k, p in enumerate((\"bank\", \"ecorecon\")):\n            v = [100 * sens[f\"C{c}_{p}\"][\"acc\"][0] if f\"C{c}_{p}\" in sens else np.nan for c in cs]\n            e = [100 * sens[f\"C{c}_{p}\"][\"acc\"][1] if f\"C{c}_{p}\" in sens else 0 for c in cs]\n            ax[1].bar(x + (k - 0.5) * w, v, w, yerr=e, color=[OI[5], OI[6]][k], capsize=2, label=NAME[p])\n        ax[1].set_xticks(x); ax[1].set_xticklabels([f\"C = {c}\" for c in cs]); ax[1].set_ylabel(\"Accuracy (%)\")\n        lo = min(100 * sens[f\"C{c}_{p}\"][\"acc\"][0] for c in cs for p in (\"bank\", \"ecorecon\") if f\"C{c}_{p}\" in sens)\n        ax[1].set_ylim(max(0, lo - 8), None); ax[1].legend(frameon=False); ax[1].set_title(\"(b) Module-bank capacity\")\n    fig.tight_layout(); fig.savefig(os.path.join(FIG, \"fig4_sensitivity.png\")); plt.close(fig)\nprint(\"figures written\")\n",
"fig1_diagram.py": "\"\"\"Fig. 1: EcoReCon-AI controller diagram.\"\"\"\nimport os\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch, FancyArrowPatch\n\nplt.rcParams.update({\"font.family\": \"serif\", \"font.serif\": [\"Times New Roman\", \"DejaVu Serif\"], \"font.size\": 7,\n                     \"mathtext.fontset\": \"stix\"})\nfig, ax = plt.subplots(figsize=(7.2, 3.3)); ax.set_xlim(0, 100); ax.set_ylim(0, 46); ax.axis(\"off\")\n\n\ndef box(x, y, w, h, txt, fc=\"#FFFFFF\", bold=False):\n    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=\"round,pad=0.3,rounding_size=1.2\", fc=fc, ec=\"#333333\", lw=0.8))\n    ax.text(x + w / 2, y + h / 2, txt, ha=\"center\", va=\"center\", fontweight=\"bold\" if bold else \"normal\", linespacing=1.3)\n\n\ndef arr(x0, y0, x1, y1, txt=None, ox=0, oy=1.0, ha=\"center\"):\n    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle=\"-|>\", mutation_scale=8, lw=0.8, color=\"#333333\"))\n    if txt: ax.text((x0 + x1) / 2 + ox, (y0 + y1) / 2 + oy, txt, ha=ha, va=\"bottom\", fontsize=6.5, style=\"italic\")\n\n\nY = 33; H = 10\nbox(1, Y, 16, H, \"Hourly unlabelled\\nbatch $X_t$ and\\ndelayed labels $L_t$\", fc=\"#F2F2F2\")\nbox(21, Y, 15, H, \"Frozen trunk $f_T$\\n(stem, stages 1\u20132)\", fc=\"#DCEBF7\")\nbox(40, Y, 15, H, \"Signature $s(X_t)$\\nchannel mean and\\nlog-std (128-d)\", fc=\"#DCEBF7\")\nbox(59, Y, 18, H, \"Nearest bank entry\\n$k^*=\\\\arg\\\\min_k\\\\, d(s, s_k)$\", fc=\"#FFF2CC\")\nbox(81, Y, 18, H, \"REUSE $\\\\theta_{k^*}$\\nno training\", fc=\"#D9EAD3\", bold=True)\nbox(59, 16, 18, 10, \"Serve $\\\\theta_{k^*}$ + BN recal.;\\nexpansion job collects\\n$L_t$ until $N_{min}=128$\", fc=\"#FCE5CD\")\nbox(81, 21.5, 18, 8, \"ADAPT NOW\\n(Bank, $D=0$)\", fc=\"#F4CCCC\", bold=True)\nbox(81, 10, 18, 9, \"DEFER to cleanest\\nforecast hour in $[t, t+D]$\\n(EcoReCon)\", fc=\"#F4CCCC\", bold=True)\nbox(59, 1, 18, 8, \"Carbon forecast\\n$CI_{u-24}$ (India trace)\", fc=\"#F2F2F2\")\nbox(5, 12, 40, 12, \"Module bank $B$ (capacity $C$, LRU eviction)\\n$\\\\theta_k$ = stage 3 + head + trunk BN statistics\\nentry 0 = pre-trained model (never evicted)\", fc=\"#EAD1DC\")\narr(17, Y + H / 2, 21, Y + H / 2); arr(36, Y + H / 2, 40, Y + H / 2); arr(55, Y + H / 2, 59, Y + H / 2)\narr(77, Y + H / 2, 81, Y + H / 2, \"$d\\\\leq\\\\tau$\", oy=0.6)\narr(68, Y, 68, 26, \"$d>\\\\tau$\", ox=1.2, oy=-0.5, ha=\"left\")\narr(77, 23, 81, 25.5); arr(77, 19, 81, 15.5)\narr(77, 5, 90, 10)\narr(59, 21, 45, 19, \"fine-tune module,\\nEXPAND bank\", oy=0.3)\narr(25, 24, 60, 33, \"stored signatures $s_k$\", ox=-6, oy=0.5)\nout = os.path.join(os.path.dirname(os.path.abspath(__file__)), \"figs\"); os.makedirs(out, exist_ok=True)\nfig.savefig(os.path.join(out, \"fig1_ecorecon.png\"), dpi=300, bbox_inches=\"tight\")\nprint(\"ok\")\n"
}
for name, text in SOURCES.items():
    open(os.path.join(WORK, name), "w", encoding="utf-8").write(text)
os.chdir(WORK); print("written:", sorted(SOURCES))


In [ ]:
# 3. Data: CIFAR-10 (official tarball, md5-checked) and the India hourly carbon-intensity trace (NASA POWER)
import urllib.request
tgz = os.path.join(DATA, "cifar-10-python.tar.gz")
if not os.path.exists(tgz):
    urllib.request.urlretrieve("https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", tgz)
assert hashlib.md5(open(tgz, "rb").read()).hexdigest() == "c58f30108f718f92721af3b95e74349a", "CIFAR md5 mismatch"
pj = os.path.join(DATA, "nasa_power_hourly_Nagpur_2023.json")
if not os.path.exists(pj):
    url = ("https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DWN&community=RE"
           "&longitude=79.09&latitude=21.15&start=20230101&end=20231231&format=JSON&time-standard=LST")
    for a in range(4):
        try:
            urllib.request.urlretrieve(url, pj); break
        except Exception as e:
            print("retry", e); time.sleep(15)
sh("python carbon.py")
import numpy as np
ci = np.load(os.path.join(DATA, "ci_india_2023.npy")); print("carbon trace: hours", ci.shape[0], "mean", round(ci.mean(), 4), "(expected 0.727)")
from common import load_cifar; x = load_cifar(); print("CIFAR train", x[0].shape, "test", x[2].shape)


In [ ]:
# 4. Pre-train the base network (skipped if finished; resumes from the last saved epoch otherwise)
if os.path.exists(os.path.join(RESULTS, "base_seed0.pt")) and os.path.exists(os.path.join(RESULTS, "pretrain_seed0.json")):
    print("base model found - skipping pre-training")
else:   # resumes automatically from MyDrive/EcoReCon_results/ckpt_pretrain_seed0.pt if a checkpoint exists
    sh("python pretrain.py 0")
print(json.load(open(os.path.join(RESULTS, "pretrain_seed0.json")))["log"][-1])


In [ ]:
# 5. Clean-data threshold tau and context-separability check (no test labels used for tau)
sh("python diag_tau.py")
d = json.load(open(os.path.join(RESULTS, "diag_tau.json")))
print("tau =", round(d["tau"], 4), "| max within-context =", round(max(d["within_max"].values()), 4),
      "| min between-context =", round(min(d["between_min"].values()), 4))
print("SEPARABLE" if max(d["within_max"].values()) < d["tau"] < min(d["between_min"].values()) else "WARNING: contexts not separable at tau")


In [ ]:
# 6. All stream runs (35 main + 15 sensitivity). Each run is saved to Drive as soon as it finishes; re-running skips finished runs.
sh("python make_jobs.py && python run_all.py")


In [ ]:
# 7. Tables + figures, then zip and download to your computer
sh("python aggregate.py && python fig1_diagram.py")
shutil.copy(os.path.join(WORK, "figs", "fig1_ecorecon.png"), os.path.join(RESULTS, "figs", "fig1_ecorecon.png"))
zp = shutil.make_archive("/content/EcoReCon_results", "zip", RESULTS)
shutil.copy(zp, os.path.join(os.path.dirname(RESULTS), "EcoReCon_results.zip")) if RESULTS.startswith("/content/drive") else None
print(open(os.path.join(RESULTS, "table_main.md")).read())
from google.colab import files
files.download(zp)
